In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
from glob import glob

import os

In [ ]:
from pathlib import Path
import os

def _find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "README.md").exists() and (candidate / "notebooks").is_dir():
            return candidate
    return current

PROJECT_ROOT = _find_project_root()
DATA_ROOT = Path(os.environ.get("GANGA_DATA_ROOT", PROJECT_ROOT / "data")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("GANGA_OUTPUT_ROOT", PROJECT_ROOT / "outputs")).expanduser()

RAW_DATA_DIR = Path(os.environ.get("GANGA_CWC_RAW_DIR", DATA_ROOT / "cwc" / "01_Raw_Data")).expanduser()
STATION_FILE = Path(os.environ.get("GANGA_CWC_STATION_FILE", DATA_ROOT / "cwc" / "02_Processed_Data" / "Ganga_CWC_Master_Stations.csv")).expanduser()
OUTPUT_DIR = OUTPUT_ROOT / "cwc"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"CWC raw data directory: {RAW_DATA_DIR}")
print(f"CWC output directory: {OUTPUT_DIR}")


In [ ]:
#find every csv 
csv_files = []

for file in RAW_DATA_DIR.rglob("*.csv"):

    # Skip empty placeholder folders
    if "Future_Empty_Files" in str(file):
        continue

    csv_files.append(str(file))

print(f"Total CSV Files Found : {len(csv_files)}")

In [ ]:
#Analyze All CSV Structures
summary = []

for file in csv_files:

    try:

        df = pd.read_csv(file, nrows=5, low_memory=False)

        summary.append({

            "File_Name": Path(file).name,
            "Category": Path(file).parts[-3],
            "Period": Path(file).parts[-2],
            "Rows_Checked": len(df),
            "Columns": len(df.columns),
            "Column_Names": list(df.columns)

        })

    except Exception as e:

        summary.append({

            "File_Name": Path(file).name,
            "Category": Path(file).parts[-3],
            "Period": Path(file).parts[-2],
            "Rows_Checked": "ERROR",
            "Columns": "ERROR",
            "Column_Names": str(e)

        })

summary = pd.DataFrame(summary)

print(f"Total Files Analysed : {len(summary)}")
summary.head()

In [ ]:
summary["Columns"].value_counts().sort_index()

In [ ]:
#Compare Column Names
unique_structures = (
    summary["Column_Names"]
    .apply(tuple)
    .value_counts()
)

print(f"Unique Column Structures : {len(unique_structures)}")

unique_structures.head(10)

In [ ]:
summary.to_csv(
    OUTPUT_DIR / "CWC_File_Structure_Report.csv",
    index=False
)

print("Inspection report saved successfully.")

In [ ]:
#Identify Structure Types because the file format are not same
structure_summary = (
    summary.groupby(["Category", "Columns"])
           .size()
           .reset_index(name="File_Count")
)

structure_summary

In [ ]:
for category in summary["Category"].unique():

    print("\n" + "="*80)
    print(f"CATEGORY : {category}")

    sample = summary[summary["Category"] == category].iloc[0]

    print("\nColumns:\n")
    for col in sample["Column_Names"]:
        print(col)

In [ ]:
#Create Category-wise Masters

from pathlib import Path

categories = ["CWC_Biological", "CWC_Chemical", "CWC_Physical"]

for category in categories:

    print("=" * 80)
    print(f"Processing : {category}")

    files = [
        f for f in csv_files
        if category in str(f)
    ]

    dfs = []

    for file in files:

        df = pd.read_csv(file, low_memory=False)

        # Extra metadata
        df["Source_File"] = Path(file).name

        if "1961_2020" in str(file):
            df["Period"] = "1961_2020"
        else:
            df["Period"] = "2021_2025"

        dfs.append(df)

    master = pd.concat(dfs, ignore_index=True, sort=False)

    output_file = OUTPUT_DIR / f"{category}_Master.csv"

    master.to_csv(output_file, index=False)

    print(f"Rows    : {len(master):,}")
    print(f"Columns : {len(master.columns)}")
    print(f"Saved   : {output_file.name}")

In [ ]:
for category in categories:

    file = OUTPUT_DIR / f"{category}_Master.csv"

    df = pd.read_csv(file, low_memory=False)

    print("=" * 80)
    print(category)
    print(df.shape)

In [ ]:
#Data Quality Report
def dataset_report(df, name):

    print("=" * 100)
    print(name)
    print("=" * 100)

    print(f"Rows                : {len(df):,}")
    print(f"Columns             : {len(df.columns)}")
    print(f"Duplicate Rows      : {df.duplicated().sum():,}")

    print("\nMissing Values")

    missing = (
        df.isna()
          .sum()
          .sort_values(ascending=False)
    )

    print(missing[missing > 0].head(20))

    print("\nData Types")

    print(df.dtypes.value_counts())

In [ ]:
bio = pd.read_csv(
    OUTPUT_DIR / "CWC_Biological_Master.csv",
    low_memory=False
)

chem = pd.read_csv(
    OUTPUT_DIR / "CWC_Chemical_Master.csv",
    low_memory=False
)

phys = pd.read_csv(
    OUTPUT_DIR / "CWC_Physical_Master.csv",
    low_memory=False
)

dataset_report(bio, "Biological")
dataset_report(chem, "Chemical")
dataset_report(phys, "Physical")

In [ ]:
for name, df in {

    "Biological": bio,
    "Chemical": chem,
    "Physical": phys

}.items():

    print("=" * 60)
    print(name)

    print("Unique Stations :", df["Station"].nunique())

In [ ]:
#Date Inspection
for name, df in {

    "Biological": bio,
    "Chemical": chem,
    "Physical": phys

}.items():

    print("=" * 60)
    print(name)

    print(df["Data Acquisition Time"].head())

In [ ]:
#LOADING MASTER FILE TO COMPARE AND FIND THE STATIONS INSIDE THE BUFFER
stations = pd.read_csv(STATION_FILE)

print(stations.shape)
stations.head()

In [ ]:
master_station_names = (
    stations["Station"]
    .astype(str)
    .str.strip()
    .str.upper()
)

for name, df in {

    "Biological": bio,
    "Chemical": chem,
    "Physical": phys

}.items():

    dataset_station_names = (
        df["Station"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    matched = dataset_station_names.isin(master_station_names).sum()

    print("=" * 70)
    print(name)
    print(f"Rows with matching stations : {matched:,}")
    print(f"Unique stations            : {dataset_station_names.nunique()}")

In [ ]:
bio_stations = set(
    bio["Station"]
    .astype(str)
    .str.strip()
    .str.upper()
)

master_stations = set(
    stations["Station"]
    .astype(str)
    .str.strip()
    .str.upper()
)

missing = sorted(bio_stations - master_stations)

print(f"Missing Stations : {len(missing)}")

missing[:50]

PHLE COMPARE KRRE THE NAMES OF STATIONS BUT SOME MISMATCHED... SO COMPARING COORDINATES OF STATIONS

In [ ]:
print(stations.columns.tolist())

In [ ]:
print(bio.columns.tolist())

In [ ]:
print("Master Stations")
print(stations[["Latitude", "Longitude"]].describe())

print("\nBiological")
print(bio[["Latitude", "Longitude"]].describe())

print("\nChemical")
print(chem[["Latitude", "Longitude"]].describe())

print("\nPhysical")
print(phys[["Latitude", "Longitude"]].describe())

In [ ]:
stations["Lat_Round"] = stations["Latitude"].round(4)
stations["Lon_Round"] = stations["Longitude"].round(4)

bio["Lat_Round"] = bio["Latitude"].round(4)
bio["Lon_Round"] = bio["Longitude"].round(4)

chem["Lat_Round"] = chem["Latitude"].round(4)
chem["Lon_Round"] = chem["Longitude"].round(4)

phys["Lat_Round"] = phys["Latitude"].round(4)
phys["Lon_Round"] = phys["Longitude"].round(4)

In [ ]:
master_coords = set(
    zip(stations["Lat_Round"], stations["Lon_Round"])
)

for name, df in {
    "Biological": bio,
    "Chemical": chem,
    "Physical": phys
}.items():

    coords = list(zip(df["Lat_Round"], df["Lon_Round"]))

    matched = sum(coord in master_coords for coord in coords)

    print("=" * 70)
    print(name)
    print(f"Matching rows : {matched:,}")
    print(f"Total rows    : {len(df):,}")

In [ ]:
for name, df in {
    "Biological": bio,
    "Chemical": chem,
    "Physical": phys
}.items():

    print("=" * 80)
    print(name)

    print(df["Basin"].value_counts(dropna=False).head(20))

In [ ]:
for name, df in {
    "Biological": bio,
    "Chemical": chem,
    "Physical": phys
}.items():

    print("=" * 80)
    print(name)

    print(df["River"].value_counts(dropna=False).head(30))

In [ ]:
bio_ganga = bio[
    bio["Basin"]
    .astype(str)
    .str.strip()
    .str.upper() == "GANGA"
].copy()

chem_ganga = chem[
    chem["Basin"]
    .astype(str)
    .str.strip()
    .str.upper() == "GANGA"
].copy()

phys_ganga = phys[
    phys["Basin"]
    .astype(str)
    .str.strip()
    .str.upper() == "GANGA"
].copy()

print("Biological :", bio_ganga.shape)
print("Chemical   :", chem_ganga.shape)
print("Physical   :", phys_ganga.shape)

In [ ]:
bio_ganga.to_csv(
    OUTPUT_DIR / "CWC_Biological_Ganga.csv",
    index=False
)

chem_ganga.to_csv(
    OUTPUT_DIR / "CWC_Chemical_Ganga.csv",
    index=False
)

phys_ganga.to_csv(
    OUTPUT_DIR / "CWC_Physical_Ganga.csv",
    index=False
)

print("Ganga datasets saved.")

In [ ]:
print(bio_ganga.shape)
print(chem_ganga.shape)
print(phys_ganga.shape)

In [ ]:
print(bio_ganga["Station"].nunique())
print(chem_ganga["Station"].nunique())
print(phys_ganga["Station"].nunique())

In [ ]:
for df in [bio_ganga, chem_ganga, phys_ganga]:

    df["Data Acquisition Time"] = pd.to_datetime(
        df["Data Acquisition Time"],
        format="%d-%m-%Y %H:%M",
        errors="coerce"
    )

In [ ]:
merge_keys = [
    "Station",
    "Data Acquisition Time"
]

for df in [bio_ganga, chem_ganga, phys_ganga]:

    df["Station"] = (
        df["Station"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

In [ ]:
for name, df in {
    "Biological": bio_ganga,
    "Chemical": chem_ganga,
    "Physical": phys_ganga
}.items():

    duplicates = df.duplicated(
        subset=merge_keys
    ).sum()

    print(f"{name} duplicate keys : {duplicates}")

In [ ]:
bio_duplicates = bio_ganga[
    bio_ganga.duplicated(
        subset=["Station", "Data Acquisition Time"],
        keep=False
    )
].sort_values(
    ["Station", "Data Acquisition Time"]
)

print(bio_duplicates.shape)

bio_duplicates.head(20)

In [ ]:
bio_duplicates[
    ["Station",
     "Data Acquisition Time",
     "Biochemical Oxygen Demand (mg/L)",
     "Chemical Oxygen Demand (mg/L)",
     "Fecal Coliform (MPN/100mL)",
     "Total Coliform (MPN/100mL)",
     "Source_File"]
].head(30)

In [ ]:
bio_ganga = bio_ganga.drop_duplicates().reset_index(drop=True)

In [ ]:
chem_ganga = chem_ganga.drop_duplicates().reset_index(drop=True)

In [ ]:
phys_ganga = phys_ganga.drop_duplicates().reset_index(drop=True)

In [ ]:
for name, df in {
    "Biological": bio_ganga,
    "Chemical": chem_ganga,
    "Physical": phys_ganga
}.items():

    print(name)
    print(df.duplicated(
        subset=["Station","Data Acquisition Time"]
    ).sum())

In [ ]:
bio_dup = bio_ganga[
    bio_ganga.duplicated(
        subset=["Station", "Data Acquisition Time"],
        keep=False
    )
].sort_values(["Station", "Data Acquisition Time"])

bio_dup.iloc[:10]

In [ ]:
bio_dup[
    bio_dup["Station"] == bio_dup.iloc[0]["Station"]
][
    bio_dup["Data Acquisition Time"] == bio_dup.iloc[0]["Data Acquisition Time"]
]

In [ ]:
# _id column hatao
bio_temp = bio_ganga.drop(columns=["_id"], errors="ignore")

# Ab exact duplicate count dekho
print(bio_temp.duplicated().sum())

Duplicate Groups Extract

In [ ]:
KEYS = ["Station", "Data Acquisition Time"]

bio_dup = (
    bio_ganga[
        bio_ganga.duplicated(subset=KEYS, keep=False)
    ]
    .sort_values(KEYS)
    .copy()
)

print("Duplicate rows :", len(bio_dup))
print("Duplicate groups :", bio_dup.groupby(KEYS).ngroups)

In [ ]:
KEYS = ["Station", "Data Acquisition Time"]

ignore_cols = KEYS

difference_summary = {}

for _, grp in bio_dup.groupby(KEYS):

    for col in bio_dup.columns:

        if col in ignore_cols:
            continue

        if grp[col].nunique(dropna=False) > 1:

            difference_summary[col] = difference_summary.get(col, 0) + 1

difference_df = (
    pd.DataFrame(
        difference_summary.items(),
        columns=["Column", "No_of_Duplicate_Groups"]
    )
    .sort_values("No_of_Duplicate_Groups", ascending=False)
)

difference_df

In [ ]:
group_size = (
    bio_dup
    .groupby(KEYS)
    .size()
    .value_counts()
    .sort_index()
)

group_size

In [ ]:
sample_keys = (
    bio_dup[KEYS]
    .drop_duplicates()
    .sample(5, random_state=42)
)

for i, (_, key) in enumerate(sample_keys.iterrows(), 1):

    print("=" * 100)
    print(f"GROUP {i}")

    display(
        bio_dup[
            (bio_dup["Station"] == key["Station"]) &
            (bio_dup["Data Acquisition Time"] == key["Data Acquisition Time"])
        ]
    )

In [ ]:
bio_dup.loc[
    (bio_dup["Station"] == "AGRA (J.B.)") &
    (bio_dup["Data Acquisition Time"] == "2021-01-01 09:40:00"),
    ["Source_File", "Period", "_id"]
]

In [ ]:
pd.set_option("display.max_colwidth", None)

bio_dup.loc[
    (bio_dup["Station"] == "AGRA (J.B.)") &
    (bio_dup["Data Acquisition Time"] == pd.Timestamp("2021-01-01 09:40:00")),
    ["Source_File", "Period", "_id"]
]

In [ ]:
bio_dup["Source_File"].nunique()

In [ ]:
sorted(bio_dup["Source_File"].unique())

In [ ]:
df["Source_File"] = (
    df["Source_File"]
      .str.strip()          # spaces remove
      .str.rstrip("'")      # trailing quote remove
      .str.replace(".csv.csv", ".csv", regex=False)
)

In [ ]:
comparison_cols = [
    c for c in bio_ganga.columns
    if c not in ["_id", "Source_File"]
]

bio_ganga = (
    bio_ganga
    .sort_values("Source_File")
    .drop_duplicates(subset=comparison_cols, keep="first")
    .reset_index(drop=True)
)

In [ ]:
bio_ganga.duplicated(
    subset=["Station", "Data Acquisition Time"]
).sum()

In [ ]:
KEYS = ["Station", "Data Acquisition Time"]

bio_remaining = (
    bio_ganga[
        bio_ganga.duplicated(subset=KEYS, keep=False)
    ]
    .sort_values(KEYS)
)

print("Remaining duplicate rows :", len(bio_remaining))
print("Remaining duplicate groups :", bio_remaining.groupby(KEYS).ngroups)

In [ ]:
ignore_cols = KEYS

difference_summary = {}

for _, grp in bio_remaining.groupby(KEYS):

    for col in bio_remaining.columns:

        if col in ignore_cols:
            continue

        if grp[col].nunique(dropna=False) > 1:

            difference_summary[col] = difference_summary.get(col, 0) + 1

difference_df = (
    pd.DataFrame(
        difference_summary.items(),
        columns=["Column", "No_of_Duplicate_Groups"]
    )
    .sort_values("No_of_Duplicate_Groups", ascending=False)
)

difference_df

In [ ]:
sample_key = (
    bio_remaining[KEYS]
    .drop_duplicates()
    .sample(1, random_state=42)
)

station = sample_key.iloc[0]["Station"]
dt = sample_key.iloc[0]["Data Acquisition Time"]

display(
    bio_remaining[
        (bio_remaining["Station"] == station) &
        (bio_remaining["Data Acquisition Time"] == dt)
    ]
)

In [ ]:
KEYS = ["Station", "Data Acquisition Time"]

grp = (
    bio_remaining
    .groupby(KEYS)
    .first()
    .reset_index()
)

station = grp.iloc[0]["Station"]
dt = grp.iloc[0]["Data Acquisition Time"]

pair = bio_remaining[
    (bio_remaining["Station"] == station) &
    (bio_remaining["Data Acquisition Time"] == dt)
]

for col in pair.columns:

    vals = pair[col].tolist()

    if len(set(map(str, vals))) > 1:

        print(col)
        print(vals)
        print("-"*60)

In [ ]:
bio_ganga["Tehsil"] = (
    bio_ganga["Tehsil"]
    .astype(str)
    .str.strip()
)

In [ ]:
bio_ganga["Source_File"] = (
    bio_ganga["Source_File"]
    .astype(str)
    .str.strip()
    .str.rstrip("'")
    .str.replace(".csv.csv", ".csv", regex=False)
)

In [ ]:
comparison_cols = [
    c for c in bio_ganga.columns
    if c != "_id"
]

bio_ganga = (
    bio_ganga
    .drop_duplicates(subset=comparison_cols)
    .reset_index(drop=True)
)

In [ ]:
bio_ganga.duplicated(
    subset=["Station", "Data Acquisition Time"]
).sum()

inspect the remaining 62

In [ ]:
KEYS = ["Station", "Data Acquisition Time"]

bio_last = (
    bio_ganga[
        bio_ganga.duplicated(subset=KEYS, keep=False)
    ]
    .sort_values(KEYS)
)

difference_summary = {}

for _, grp in bio_last.groupby(KEYS):

    for col in bio_last.columns:

        if col in KEYS:
            continue

        if grp[col].nunique(dropna=False) > 1:
            difference_summary[col] = difference_summary.get(col, 0) + 1

pd.DataFrame(
    difference_summary.items(),
    columns=["Column","Groups"]
).sort_values("Groups", ascending=False)

In [ ]:
sample = (
    bio_last[KEYS]
    .drop_duplicates()
    .sample(1, random_state=1)
)

st = sample.iloc[0]["Station"]
dt = sample.iloc[0]["Data Acquisition Time"]

pair = bio_last[
    (bio_last["Station"] == st) &
    (bio_last["Data Acquisition Time"] == dt)
]

for col in pair.columns:

    vals = pair[col].tolist()

    if len(set(map(str, vals))) > 1:
        print(col, vals)

In [ ]:
# ---------- Metadata Standardization ----------

bio_ganga["Station"] = (
    bio_ganga["Station"]
    .astype(str)
    .str.strip()
    .str.upper()
)

bio_ganga["District"] = (
    bio_ganga["District"]
    .astype(str)
    .str.strip()
)

bio_ganga["Tehsil"] = (
    bio_ganga["Tehsil"]
    .astype(str)
    .str.strip()
)

bio_ganga["SlNo"] = (
    bio_ganga["SlNo"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)

bio_ganga["Source_File"] = (
    bio_ganga["Source_File"]
    .astype(str)
    .str.strip()
    .str.rstrip("'")
    .str.replace(".csv.csv", ".csv", regex=False)
    .str.replace("parameters", "parameter", regex=False)
)

In [ ]:
comparison_cols = [
    c for c in bio_ganga.columns
    if c != "_id"
]

bio_ganga = (
    bio_ganga
    .drop_duplicates(subset=comparison_cols)
    .reset_index(drop=True)
)

In [ ]:
bio_ganga.duplicated(
    subset=["Station", "Data Acquisition Time"]
).sum()

In [ ]:
KEYS = ["Station", "Data Acquisition Time"]

for _, grp in bio_last.groupby(KEYS):

    diff = {}

    for col in grp.columns:
        if col in KEYS:
            continue

        vals = grp[col].astype(str).unique()

        if len(vals) > 1:
            diff[col] = vals.tolist()

    # Agar scientific columns me difference hai to print karo
    scientific_cols = [
        "Biochemical Oxygen Demand (mg/L)",
        "Chemical Oxygen Demand (mg/L)",
        "Fecal Coliform (MPN/100mL)",
        "Total Coliform (MPN/100mL)",
        "Latitude",
        "Longitude"
    ]

    if any(col in diff for col in scientific_cols):
        print("="*80)
        print(grp[KEYS].iloc[0].to_dict())
        print(diff)

In [ ]:
# ============================================================
# STEP 10A : Metadata Standardization
# ============================================================

bio_ganga["Station"] = (
    bio_ganga["Station"]
    .astype(str)
    .str.strip()
    .str.upper()
)

bio_ganga["District"] = (
    bio_ganga["District"]
    .astype(str)
    .str.strip()
)

bio_ganga["Tehsil"] = (
    bio_ganga["Tehsil"]
    .astype(str)
    .str.strip()
)

bio_ganga["SlNo"] = (
    bio_ganga["SlNo"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)

bio_ganga["Source_File"] = (
    bio_ganga["Source_File"]
    .astype(str)
    .str.strip()
    .str.rstrip("'")
    .str.replace(".csv.csv", ".csv", regex=False)
    .str.replace("parameters", "parameter", regex=False)
)

In [ ]:
# ============================================================
# STEP 10B : Remove Metadata Duplicates
# ============================================================

comparison_cols = [
    c for c in bio_ganga.columns
    if c not in ["_id"]
]

bio_ganga = (
    bio_ganga
    .drop_duplicates(subset=comparison_cols)
    .reset_index(drop=True)
)

print("Biological Shape :", bio_ganga.shape)

In [ ]:
# ============================================================
# STEP 10C : Duplicate Report
# ============================================================

KEYS = [
    "Station",
    "Data Acquisition Time"
]

dup = bio_ganga.duplicated(
    subset=KEYS
).sum()

print(f"Duplicate Station-Date Keys : {dup}")

# NOTE:
# Remaining 62 duplicate Station-Date combinations correspond
# to different monitoring locations having different coordinates.
# They are retained intentionally.

In [ ]:
# ============================================================
# STEP 10D : Save Clean Biological Dataset
# ============================================================

bio_output = OUTPUT_DIR / "CWC_Biological_Ganga_Clean.csv"

bio_ganga.to_csv(
    bio_output,
    index=False
)

print("Saved :", bio_output)
print("Rows :", len(bio_ganga))

In [ ]:
# ============================================================
# STEP 11A : Metadata Standardization (Chemical)
# ============================================================

chem_ganga["Station"] = (
    chem_ganga["Station"]
    .astype(str)
    .str.strip()
    .str.upper()
)

chem_ganga["District"] = (
    chem_ganga["District"]
    .astype(str)
    .str.strip()
)

chem_ganga["Tehsil"] = (
    chem_ganga["Tehsil"]
    .astype(str)
    .str.strip()
)

chem_ganga["SlNo"] = (
    chem_ganga["SlNo"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)

chem_ganga["Source_File"] = (
    chem_ganga["Source_File"]
    .astype(str)
    .str.strip()
    .str.rstrip("'")
    .str.replace(".csv.csv", ".csv", regex=False)
    .str.replace("parameters", "parameter", regex=False)
)

In [ ]:
# ============================================================
# STEP 11B : Remove Metadata Duplicates
# ============================================================

comparison_cols = [
    c for c in chem_ganga.columns
    if c != "_id"
]

chem_ganga = (
    chem_ganga
    .drop_duplicates(subset=comparison_cols)
    .reset_index(drop=True)
)

print("Chemical Shape :", chem_ganga.shape)

In [ ]:
# ============================================================
# STEP 11C : Duplicate Report
# ============================================================

dup = chem_ganga.duplicated(
    subset=["Station", "Data Acquisition Time"]
).sum()

print("Duplicate Station-Date :", dup)

In [ ]:
KEYS = ["Station", "Data Acquisition Time"]

chem_last = (
    chem_ganga[
        chem_ganga.duplicated(subset=KEYS, keep=False)
    ]
    .sort_values(KEYS)
)

display(chem_last)

In [ ]:
for _, grp in chem_last.groupby(KEYS):

    diff = {}

    for col in grp.columns:

        if col in KEYS:
            continue

        vals = grp[col].astype(str).unique()

        if len(vals) > 1:
            diff[col] = vals.tolist()

    print("="*100)
    print(grp[KEYS].iloc[0].to_dict())
    print(diff)

In [ ]:
# ============================================================
# STEP 11D : Save Clean Chemical Dataset
# ============================================================

chem_output = OUTPUT_DIR / "CWC_Chemical_Ganga_Clean.csv"

chem_ganga.to_csv(
    chem_output,
    index=False
)

print("Saved :", chem_output)
print("Rows :", len(chem_ganga))

In [ ]:
# ============================================================
# STEP 12A : Metadata Standardization (Physical)
# ============================================================

phys_ganga["Station"] = (
    phys_ganga["Station"]
    .astype(str)
    .str.strip()
    .str.upper()
)

phys_ganga["District"] = (
    phys_ganga["District"]
    .astype(str)
    .str.strip()
)

phys_ganga["Tehsil"] = (
    phys_ganga["Tehsil"]
    .astype(str)
    .str.strip()
)

phys_ganga["SlNo"] = (
    phys_ganga["SlNo"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)

phys_ganga["Source_File"] = (
    phys_ganga["Source_File"]
    .astype(str)
    .str.strip()
    .str.rstrip("'")
    .str.replace(".csv.csv", ".csv", regex=False)
    .str.replace("parameters", "parameter", regex=False)
)

In [ ]:
# ============================================================
# STEP 12B : Remove Metadata Duplicates
# ============================================================

comparison_cols = [
    c for c in phys_ganga.columns
    if c != "_id"
]

phys_ganga = (
    phys_ganga
    .drop_duplicates(subset=comparison_cols)
    .reset_index(drop=True)
)

print("Physical Shape :", phys_ganga.shape)

In [ ]:
# ============================================================
# STEP 12C : Duplicate Report
# ============================================================

dup = phys_ganga.duplicated(
    subset=["Station", "Data Acquisition Time"]
).sum()

print("Duplicate Station-Date :", dup)

In [ ]:
KEYS = ["Station", "Data Acquisition Time"]

phys_last = (
    phys_ganga[
        phys_ganga.duplicated(subset=KEYS, keep=False)
    ]
    .sort_values(KEYS)
)

display(phys_last)

In [ ]:
for _, grp in phys_last.groupby(KEYS):

    diff = {}

    for col in grp.columns:

        if col in KEYS:
            continue

        vals = grp[col].astype(str).unique()

        if len(vals) > 1:
            diff[col] = vals.tolist()

    print("=" * 100)
    print(grp[KEYS].iloc[0].to_dict())
    print(diff)

In [ ]:
# ============================================================
# STEP 12E : Save Clean Physical Dataset
# ============================================================

phys_output = OUTPUT_DIR / "CWC_Physical_Ganga_Clean.csv"

phys_ganga.to_csv(
    phys_output,
    index=False
)

print("Saved :", phys_output)
print("Rows :", len(phys_ganga))